In [2]:
import numpy as np
from pathlib import Path

file_path = Path(
    "./pretrain_process_dataset/GTT23-Swallow-origin/dataset.npy"
)

print("文件大小:", round(file_path.stat().st_size / 1024**3, 2), "GB")

with open(file_path, "rb") as f:
    version = np.lib.format.read_magic(f)
    print("NPY 版本:", version)

    if version == (1, 0):
        shape, fortran_order, dtype = np.lib.format.read_array_header_1_0(f)
    else:
        shape, fortran_order, dtype = np.lib.format.read_array_header_2_0(f)

    print("Shape:", shape)
    print("Dtype:", dtype)
    print("Fortran order:", fortran_order)
    print("数据起始偏移:", f.tell(), "bytes")

print("文件头读取完成，没有加载完整数据。")

文件大小: 26.91 GB
NPY 版本: (1, 0)
Shape: (7222474, 2, 1000)
Dtype: uint16
Fortran order: False
数据起始偏移: 128 bytes
文件头读取完成，没有加载完整数据。


不是已经进行了Swallow的数据增强。而是转为其CIF一致性特征，也就是TAM窗口不就是两个通道，上下行嘛

In [4]:

import numpy as np
from pathlib import Path

# ==================== 配置区域 ====================
file_path = "./pretrain_process_dataset/GTT23-Swallow-origin/dataset.npy"
show_samples = 3
show_elements = 30
# ==================================================


def view_npy(file_path):
    path = Path(file_path)

    if not path.exists():
        print(f"文件不存在：{path.resolve()}")
        return

    print(f"文件路径：{path.resolve()}")
    print(f"文件大小：{path.stat().st_size / 1024**3:.2f} GB")

    print("\n正在以只读内存映射方式打开文件……")
    data = np.load(path, mmap_mode="r", allow_pickle=False)

    print("\n========== 基本信息 ==========")
    print("数据形状：", data.shape)
    print("数据维度：", data.ndim)
    print("数据类型：", data.dtype)
    print("样本数量：", data.shape[0])
    print("每个样本形状：", data.shape[1:])
    print("总元素数量：", f"{data.size:,}")
    print("理论数据大小：", f"{data.nbytes / 1024**3:.2f} GB")

    print("\n========== 样本预览 ==========")

    for i in range(min(show_samples, data.shape[0])):
        print(f"\n第 {i} 个样本：")
        sample = data[i]

        print("样本形状：", sample.shape)

        for channel in range(sample.shape[0]):
            values = sample[channel, :show_elements]
            print(f"  通道 {channel} 前 {show_elements} 个值：")
            print(" ", values)

    print("\n========== 通道统计 ==========")

    # 只扫描前 10000 个样本，避免读取整个大文件
    n = min(10000, data.shape[0])
    subset = data[:n]

    for channel in range(data.shape[1]):
        values = subset[:, channel, :]

        print(f"\n通道 {channel}（前 {n} 个样本）：")
        print("最小值：", values.min())
        print("最大值：", values.max())
        print("均值：", values.mean())

    print("\n查看完成。")

    
view_npy(file_path)

文件路径：/mnt/d/workspace/WF-SCSM/pretrain_process_dataset/GTT23-Swallow-origin/dataset.npy
文件大小：26.91 GB

正在以只读内存映射方式打开文件……

========== 基本信息 ==========
数据形状： (7222474, 2, 1000)
数据维度： 3
数据类型： uint16
样本数量： 7222474
每个样本形状： (2, 1000)
总元素数量： 14,444,948,000
理论数据大小： 26.91 GB

========== 样本预览 ==========

第 0 个样本：
样本形状： (2, 1000)
  通道 0 前 30 个值：
  [6 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
  通道 1 前 30 个值：
  [0 2 1 1 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

第 1 个样本：
样本形状： (2, 1000)
  通道 0 前 30 个值：
  [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0]
  通道 1 前 30 个值：
  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]

第 2 个样本：
样本形状： (2, 1000)
  通道 0 前 30 个值：
  [1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 5 0 7 3 2 0 0 1 3 0 5 0]
  通道 1 前 30 个值：
  [  0   0   0   0   1   0   0  22   0   0   0   0   0   0   0   0   0   0
   1  10  20 309   9   0  50  14 107  47  18   0]

========== 通道统计 ==========

通道 0（前 10000 个样本）：
最小值： 0
最大值： 215
均值： 0.1275162

通道 1（前 1